In [17]:
print("hey")

hey


FR-01 → Data Loading + Validation

In [18]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

print("All libraries imported successfully!")

All libraries imported successfully!


Getting Data

In [19]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
stores = pd.read_csv("data/stores.csv")
oil = pd.read_csv("data/oil.csv")
holidays = pd.read_csv("data/holidays_events.csv")
transactions = pd.read_csv("data/transactions.csv")

In [20]:
print("train:", train.shape)
print("test:", test.shape)
print("stores:", stores.shape)
print("oil:", oil.shape)
print("holidays:", holidays.shape)
print("transactions:", transactions.shape)

train: (3000888, 6)
test: (28512, 5)
stores: (54, 5)
oil: (1218, 2)
holidays: (350, 6)
transactions: (83488, 3)


In [21]:
#create the date col before merging
train['date'] = pd.to_datetime(train['date'])
test['date'] = pd.to_datetime(test['date'])
oil['date'] = pd.to_datetime(oil['date'])
holidays['date'] = pd.to_datetime(holidays['date'])
transactions['date'] = pd.to_datetime(transactions['date'])

In [22]:
#merge dataset
df = train.copy()

df = df.merge(stores, on="store_nbr", how="left")
df = df.merge(oil, on="date", how="left")
df = df.merge(transactions, on=["store_nbr", "date"], how="left")

# remove duplicate holiday dates
holidays = holidays.drop_duplicates(subset=['date'])

# merge holidays
df = df.merge(holidays, on="date", how="left")

In [23]:
#confusing names
df = df.rename(columns={
    "type_x": "store_type",
    "type_y": "holiday_type"
})

In [24]:
#merge diagonistic 
print("Merge Null Diagnostics:")
print(df.isnull().sum())

Merge Null Diagnostics:
id                    0
date                  0
store_nbr             0
family                0
sales                 0
onpromotion           0
city                  0
state                 0
store_type            0
cluster               0
dcoilwtico       928422
transactions     245784
holiday_type    2551824
locale          2551824
locale_name     2551824
description     2551824
transferred     2551824
dtype: int64


In [25]:
required_columns = ['store_nbr','family','date','sales']

for col in required_columns:
    if col not in df.columns:
        print(f"Missing column: {col}")

In [26]:
print("Data Types:")
print(df.dtypes)

Data Types:
id                       int64
date            datetime64[us]
store_nbr                int64
family                     str
sales                  float64
onpromotion              int64
city                       str
state                      str
store_type                 str
cluster                  int64
dcoilwtico             float64
transactions           float64
holiday_type               str
locale                     str
locale_name                str
description                str
transferred             object
dtype: object


In [27]:
negative_sales = df[df['sales'] < 0]

print("Negative sales rows:", len(negative_sales))

Negative sales rows: 0


In [28]:
null_report = df.isnull().sum().sort_values(ascending=False)

print("Null Count Report:")
print(null_report)

Null Count Report:
locale          2551824
holiday_type    2551824
locale_name     2551824
description     2551824
transferred     2551824
dcoilwtico       928422
transactions     245784
id                    0
date                  0
store_type            0
state                 0
city                  0
onpromotion           0
sales                 0
family                0
store_nbr             0
cluster               0
dtype: int64


In [29]:
print("Merged Dataset Shape:", df.shape)

Merged Dataset Shape: (3000888, 17)


In [30]:
stores_count = df['store_nbr'].nunique()
family_count = df['family'].nunique()
days_count = df['date'].nunique()

expected_rows = stores_count * family_count * days_count

print("Expected Rows:", expected_rows)
print("Actual Rows:", len(df))

Expected Rows: 3000888
Actual Rows: 3000888


In [31]:
if expected_rows != len(df):
    print("WARNING: Row count mismatch detected")

FR-02 → Data Preprocessing

Missing oil price values were imputed using a forward-fill strategy to propagate the last known trading value across non-trading days. A backward-fill step was additionally applied to handle missing values at the beginning of the dataset, resulting in 0% null values in the dcoilwtico feature.

In [32]:
# fill oil prices
df['dcoilwtico'] = df['dcoilwtico'].ffill().bfill()

# verify
print("Oil price nulls:", df['dcoilwtico'].isnull().sum())


Oil price nulls: 0


Handle Transferred Holidays
The dataset has holidays that were officially moved to another date.

In [33]:
# create holiday flag
df['is_holiday'] = False

# mark holidays that are NOT transferred
df.loc[
    (df['holiday_type'].notna()) & (df['transferred'] == False),
    'is_holiday'
] = True

print(df['is_holiday'].value_counts())

is_holiday
False    2567862
True      433026
Name: count, dtype: int64


Outlier detection using IQR

In [34]:
def detect_outliers(group):
    
    Q1 = group['sales'].quantile(0.25)
    Q3 = group['sales'].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    group['is_outlier'] = (group['sales'] < lower) | (group['sales'] > upper)
    
    return group

df = df.groupby(['store_nbr','family'], group_keys=False).apply(detect_outliers)

print("Outliers detected:", df['is_outlier'].sum())

Outliers detected: 113422


In [35]:
print(df.columns)

Index(['id', 'date', 'sales', 'onpromotion', 'city', 'state', 'store_type',
       'cluster', 'dcoilwtico', 'transactions', 'holiday_type', 'locale',
       'locale_name', 'description', 'transferred', 'is_holiday',
       'is_outlier'],
      dtype='str')


In [36]:
# Drop duplicate/reset columns from previous steps
df = df.drop(columns=['level_0','index'], errors='ignore')

# Check the columns
print(df.columns)

Index(['id', 'date', 'sales', 'onpromotion', 'city', 'state', 'store_type',
       'cluster', 'dcoilwtico', 'transactions', 'holiday_type', 'locale',
       'locale_name', 'description', 'transferred', 'is_holiday',
       'is_outlier'],
      dtype='str')


In [37]:
# restore the missing columns from original train
df[['store_nbr','family']] = train[['store_nbr','family']]

# check columns
print(df.columns)

Index(['id', 'date', 'sales', 'onpromotion', 'city', 'state', 'store_type',
       'cluster', 'dcoilwtico', 'transactions', 'holiday_type', 'locale',
       'locale_name', 'description', 'transferred', 'is_holiday', 'is_outlier',
       'store_nbr', 'family'],
      dtype='str')


In [38]:
df = df.sort_values(by=['store_nbr','family','date'])
print("Data sorted successfully")

Data sorted successfully


In [39]:
df.head()

,id,date,sales,onpromotion,city,state,store_type,cluster,dcoilwtico,transactions,holiday_type,locale,locale_name,description,transferred,is_holiday,is_outlier,store_nbr,family
0,0,2013-01-01,0.0,0,Quito,Pichincha,D,13,93.14,NaN,Holiday,National,Ecuador,Primer dia del ano,False,True,False,1,AUTOMOTIVE
1782,1782,2013-01-02,2.0,0,Quito,Pichincha,D,13,93.14,2111.0,NaN,NaN,NaN,NaN,NaN,False,False,1,AUTOMOTIVE
3564,3564,2013-01-03,3.0,0,Quito,Pichincha,D,13,92.97,1833.0,NaN,NaN,NaN,NaN,NaN,False,False,1,AUTOMOTIVE
5346,5346,2013-01-04,3.0,0,Quito,Pichincha,D,13,93.12,1863.0,NaN,NaN,NaN,NaN,NaN,False,False,1,AUTOMOTIVE
7128,7128,2013-01-05,5.0,0,Quito,Pichincha,D,13,93.12,1509.0,Work Day,National,Ecuador,Recupero puente Navidad,False,True,False,1,AUTOMOTIVE


In [40]:
row_count = len(df)

with open("preprocessing_log.txt","w") as f:
    f.write(f"Final row count after preprocessing: {row_count}\n")

print("Row count logged:", row_count)

Row count logged: 3000888


Validation for FR-02

In [41]:
print("Oil price nulls:", df['dcoilwtico'].isnull().sum())
print("Holiday flag counts:")
print(df['is_holiday'].value_counts())

print("Outliers detected:", df['is_outlier'].sum())

print("Dataset shape:", df.shape)

Oil price nulls: 0
Holiday flag counts:
is_holiday
False    2567862
True      433026
Name: count, dtype: int64
Outliers detected: 113422
Dataset shape: (3000888, 19)


Conclusion: The dataset is now fully cleaned, flagged, and temporally ordered, ready for feature engineering (FR-03).

FR-03 Feature Engineering Module

In [42]:
import pandas as pd
import numpy as np

class FastFeatureEngineer:
    def __init__(self, df, transactions=None, oil_price=None, holidays=None, store_meta=None):
        self.df = df.copy()
        self.transactions = transactions
        self.oil_price = oil_price
        self.holidays = holidays
        self.store_meta = store_meta

    def add_temporal_features(self):
        self.df['day_of_week'] = self.df['date'].dt.dayofweek
        self.df['day_of_month'] = self.df['date'].dt.day
        self.df['week_of_year'] = self.df['date'].dt.isocalendar().week
        self.df['month'] = self.df['date'].dt.month
        self.df['quarter'] = self.df['date'].dt.quarter
        self.df['year'] = self.df['date'].dt.year
        self.df['is_weekend'] = self.df['day_of_week'].isin([5,6]).astype(np.uint8)
        return self

    def add_lag_and_rolling(self, lags=[1,7,14,365], windows=[7,14,28]):
        gb = self.df.groupby(['store_nbr','family'], sort=False)
        # Lag features
        for lag in lags:
            self.df[f'sales_lag_{lag}d'] = gb['sales'].shift(lag)
        # Rolling features
        for w in windows:
            self.df[f'sales_roll_mean_{w}d'] = gb['sales'].transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
            self.df[f'sales_roll_std_{w}d'] = gb['sales'].transform(lambda x: x.shift(1).rolling(w, min_periods=1).std())
        return self

    def add_onpromotion_features(self):
        gb = self.df.groupby(['store_nbr','family'], sort=False)
        self.df['onpromotion_lag_1d'] = gb['onpromotion'].shift(1)
        self.df['onpromotion_rolling_7d'] = gb['onpromotion'].transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
        return self

    def add_macroeconomic_features(self):
        if self.oil_price is not None:
            self.df = self.df.merge(self.oil_price, on='date', how='left')
            self.df['dcoilwtico_lag_7d'] = self.df['dcoilwtico'].shift(7)
            self.df['dcoilwtico_rolling_28d'] = self.df['dcoilwtico'].shift(1).rolling(28, min_periods=1).mean()
        return self

    def add_transaction_features(self):
        if self.transactions is not None:
            self.df = self.df.merge(self.transactions, on=['date','store_nbr'], how='left')
            self.df['transactions_lag_7d'] = self.df.groupby('store_nbr')['transactions'].shift(7)
        return self

    def add_store_metadata(self):
        if self.store_meta is not None:
            self.df = self.df.merge(self.store_meta, on='store_nbr', how='left')
            self.df['store_type'] = self.df['store_type'].astype('category').cat.codes
        return self

    def add_cannibalization_features(self, top_n=3):
        stores = self.df['store_nbr'].unique()
        cannibal_features = []
        
        for s in stores:
            # Pivot by date and family for this store
            df_store = self.df[self.df['store_nbr']==s].pivot(index='date', columns='family', values='sales')
            
            # Correlation matrix
            corr = df_store.corr()
            
            # For each family, compute top correlated families
            for f in df_store.columns:
                top_fams = corr[f].sort_values(ascending=False).iloc[1:top_n+1].index
                df_store[f'{f}_top_corr_mean'] = df_store[top_fams].shift(1).mean(axis=1)
            
            # Add store number back as column
            df_store['store_nbr'] = s
            cannibal_features.append(df_store)
        
        cannibal_df = pd.concat(cannibal_features).reset_index()
        
        # Melt back to long format: date + store_nbr + family
        id_vars = ['date','store_nbr']
        value_vars = [c for c in cannibal_df.columns if '_top_corr_mean' in c]
        cannibal_long = cannibal_df.melt(id_vars=id_vars, value_vars=value_vars,
                                        var_name='family_feature', value_name='top_corr_mean')
        # Extract family name
        cannibal_long['family'] = cannibal_long['family_feature'].str.replace('_top_corr_mean','')
        cannibal_long = cannibal_long.drop('family_feature', axis=1)
        
        # Merge back to main DF
        self.df = self.df.merge(cannibal_long, on=['date','store_nbr','family'], how='left')
        
        return self
        

    def transform(self):
        return self.df 

In [43]:
print(stores.columns)

Index(['store_nbr', 'city', 'state', 'type', 'cluster'], dtype='str')


In [44]:
stores = stores.rename(columns={
    'type': 'store_type',   # remove trailing space if present
    'cluster': 'store_cluster'     # rename cluster to store_cluster
})

print(stores.columns)

Index(['store_nbr', 'city', 'state', 'store_type', 'store_cluster'], dtype='str')


In [45]:
print(train.columns)

Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion'], dtype='str')


In [46]:
fe = FastFeatureEngineer(
    df=train,
    transactions=transactions,
    oil_price=oil,
    holidays=holidays,
    store_meta=stores
)

train_features = (
    fe.add_temporal_features()
      .add_lag_and_rolling()
      .add_onpromotion_features()
      .add_macroeconomic_features()
      .add_transaction_features()
      .add_store_metadata()
      .add_cannibalization_features()
      .transform()
)

checking...

In [47]:
# Print all columns
print("Columns in dataset:")
print(train_features.columns.tolist())

Columns in dataset:
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'day_of_week', 'day_of_month', 'week_of_year', 'month', 'quarter', 'year', 'is_weekend', 'sales_lag_1d', 'sales_lag_7d', 'sales_lag_14d', 'sales_lag_365d', 'sales_roll_mean_7d', 'sales_roll_std_7d', 'sales_roll_mean_14d', 'sales_roll_std_14d', 'sales_roll_mean_28d', 'sales_roll_std_28d', 'onpromotion_lag_1d', 'onpromotion_rolling_7d', 'dcoilwtico', 'dcoilwtico_lag_7d', 'dcoilwtico_rolling_28d', 'transactions', 'transactions_lag_7d', 'city', 'state', 'store_type', 'store_cluster', 'top_corr_mean']


In [48]:
# Look at the first 20 rows
print(train_features.head(20))

    id       date  store_nbr               family  sales  onpromotion  \
0    0 2013-01-01          1           AUTOMOTIVE    0.0            0   
1    1 2013-01-01          1            BABY CARE    0.0            0   
2    2 2013-01-01          1               BEAUTY    0.0            0   
3    3 2013-01-01          1            BEVERAGES    0.0            0   
4    4 2013-01-01          1                BOOKS    0.0            0   
5    5 2013-01-01          1         BREAD/BAKERY    0.0            0   
6    6 2013-01-01          1          CELEBRATION    0.0            0   
7    7 2013-01-01          1             CLEANING    0.0            0   
8    8 2013-01-01          1                DAIRY    0.0            0   
9    9 2013-01-01          1                 DELI    0.0            0   
10  10 2013-01-01          1                 EGGS    0.0            0   
11  11 2013-01-01          1         FROZEN FOODS    0.0            0   
12  12 2013-01-01          1            GROCERY I  

In [49]:
# Count NaNs per column
print(train_features.isna().sum())

id                             0
date                           0
store_nbr                      0
family                         0
sales                          0
onpromotion                    0
day_of_week                    0
day_of_month                   0
week_of_year                   0
month                          0
quarter                        0
year                           0
is_weekend                     0
sales_lag_1d                1782
sales_lag_7d               12474
sales_lag_14d              24948
sales_lag_365d            650430
sales_roll_mean_7d          1782
sales_roll_std_7d           3564
sales_roll_mean_14d         1782
sales_roll_std_14d          3564
sales_roll_mean_28d         1782
sales_roll_std_28d          3564
onpromotion_lag_1d          1782
onpromotion_rolling_7d      1782
dcoilwtico                928422
dcoilwtico_lag_7d         928429
dcoilwtico_rolling_28d    921727
transactions              245784
transactions_lag_7d       246162
city      

In [50]:
# Pick one store-family and check
tmp = train_features[(train_features['store_nbr']==1) & (train_features['family']=='FOODS')]
print(tmp[['date','sales','sales_lag_7d','sales_roll_mean_7d']].head(15))

Empty DataFrame
Columns: [date, sales, sales_lag_7d, sales_roll_mean_7d]
Index: []


In [51]:
train_features = train_features.dropna() #dropping nan

FR-04 EDA

In [53]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

# Make output directory
os.makedirs("outputs/eda", exist_ok=True)

# 1️⃣ Time-series decomposition for 3 store-family combinations
store_family_samples = train_features.groupby(['store_nbr','family']).size().sample(3, random_state=42).index

for store_nbr, family in store_family_samples:
    ts = train_features[(train_features['store_nbr']==store_nbr) & (train_features['family']==family)]
    ts = ts.set_index('date').sort_index()
    
    result = seasonal_decompose(ts['sales'], model='additive', period=365)
    
    plt.figure(figsize=(12,8))
    result.plot()
    plt.suptitle(f"Time-Series Decomposition: Store {store_nbr}, Family {family}", fontsize=16)
    plt.tight_layout()
    plt.savefig(f"outputs/eda/ts_decompose_store{store_nbr}_family{family}.png")
    plt.close()

# 2️⃣ Correlation heatmap of numerical features
numeric_cols = train_features.select_dtypes(include='number').columns
plt.figure(figsize=(12,10))
sns.heatmap(train_features[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap of Numerical Features")
plt.savefig("outputs/eda/correlation_heatmap.png")
plt.close()

# 3️⃣ Distribution plots for sales and onpromotion
plt.figure(figsize=(10,5))
sns.histplot(train_features['sales'], bins=50, kde=True)
plt.title("Distribution of Sales")
plt.savefig("outputs/eda/sales_distribution.png")
plt.close()

plt.figure(figsize=(10,5))
sns.histplot(train_features['onpromotion'], bins=50, kde=True)
plt.title("Distribution of OnPromotion")
plt.savefig("outputs/eda/onpromotion_distribution.png")
plt.close()

# 4️⃣ Holiday lift analysis
train_features['holiday_type'] = train_features['date'].map(
    holidays.set_index('date')['type']
).fillna('non_holiday')

plt.figure(figsize=(10,5))
sns.barplot(x='holiday_type', y='sales', data=train_features.groupby('holiday_type')['sales'].mean().reset_index())
plt.title("Holiday Lift Analysis")
plt.ylabel("Mean Sales")
plt.savefig("outputs/eda/holiday_lift.png")
plt.close()

# 5️⃣ Oil price vs aggregate sales scatter plot
daily_sales = train_features.groupby('date')['sales'].sum().reset_index()
daily_data = daily_sales.merge(oil, on='date', how='left')

plt.figure(figsize=(12,6))
sns.scatterplot(x='dcoilwtico', y='sales', data=daily_data)
plt.title("Oil Price vs Aggregate Sales")
plt.xlabel("Oil Price (dcoilwtico)")
plt.ylabel("Total Sales")
plt.savefig("outputs/eda/oil_vs_sales.png")
plt.close()

# 6️⃣ Cross-family correlation matrix
pivot_sales = train_features.pivot_table(index='date', columns='family', values='sales', aggfunc='sum')
plt.figure(figsize=(12,10))
sns.heatmap(pivot_sales.corr(), cmap="coolwarm")
plt.title("Cross-Family Sales Correlation")
plt.savefig("outputs/eda/cross_family_corr.png")
plt.close()

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

Build Model

In [54]:
train["day_of_week"] = train["date"].dt.dayofweek
train["month"] = train["date"].dt.month
train["year"] = train["date"].dt.year
train["family"] = train["family"].astype("category").cat.codes

In [55]:
#choosing features
features = [
    "store_nbr",
    "family",
    "onpromotion",
    "day_of_week",
    "month",
    "year"
]

target = "sales"

Train/validation split

In [56]:
split_date = "2017-01-01"

train_data = train[train["date"] < split_date]
valid_data = train[train["date"] >= split_date]

# Create X and y
X_train = train_data[features]
y_train = train_data[target]

X_valid = valid_data[features]
y_valid = valid_data[target]

Train lightGBM

In [57]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

lgb_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=31,
    random_state=42
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="rmse",
)

lgb_pred = lgb_model.predict(X_valid)

rmse_lgb = np.sqrt(mean_squared_error(y_valid, lgb_pred))
print("LightGBM RMSE:", rmse_lgb)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.311962 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 307
[LightGBM] [Info] Number of data points in the train set: 2596374, number of used features: 6
[LightGBM] [Info] Start training from score 338.713869
LightGBM RMSE: 381.02332349308665


CatBoost Model

In [59]:
from catboost import CatBoostRegressor

cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    verbose=100
)

cat_model.fit(X_train, y_train)

cat_pred = cat_model.predict(X_valid)

rmse_cat = np.sqrt(mean_squared_error(y_valid, cat_pred))
print("CatBoost RMSE:", rmse_cat)

0:	learn: 1024.7473128	total: 600ms	remaining: 9m 59s
100:	learn: 500.7324159	total: 53.8s	remaining: 7m 58s
200:	learn: 446.9477148	total: 1m 34s	remaining: 6m 17s
300:	learn: 418.3239325	total: 2m 15s	remaining: 5m 15s
400:	learn: 398.9254099	total: 2m 59s	remaining: 4m 28s
500:	learn: 383.5123803	total: 3m 39s	remaining: 3m 38s
600:	learn: 371.6890924	total: 4m 23s	remaining: 2m 54s
700:	learn: 361.8167437	total: 5m 3s	remaining: 2m 9s
800:	learn: 353.8194760	total: 5m 42s	remaining: 1m 24s
900:	learn: 346.7316777	total: 6m 22s	remaining: 42s
999:	learn: 340.5241271	total: 7m 2s	remaining: 0us
CatBoost RMSE: 409.90258177268765


XG Boost Model

In [60]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=100
)

xgb_pred = xgb_model.predict(X_valid)

rmse_xgb = np.sqrt(mean_squared_error(y_valid, xgb_pred))
print("XGBoost RMSE:", rmse_xgb)

[0]	validation_0-rmse:1331.86849
[100]	validation_0-rmse:459.14099
[200]	validation_0-rmse:413.41602
[300]	validation_0-rmse:400.36941
[400]	validation_0-rmse:393.83040
[500]	validation_0-rmse:391.93292
[600]	validation_0-rmse:393.87866
[700]	validation_0-rmse:398.36722
[800]	validation_0-rmse:403.49946
[900]	validation_0-rmse:405.59547
[999]	validation_0-rmse:409.05681
XGBoost RMSE: 409.05681052950797


Ensemble Model (LightGBM + XGBoost + CatBoost)

In [61]:
ensemble_pred = (lgb_pred + xgb_pred + cat_pred) / 3

rmse_ensemble = np.sqrt(mean_squared_error(y_valid, ensemble_pred))

print("Ensemble RMSE:", rmse_ensemble)

Ensemble RMSE: 382.5243914215874


Compare 3 Models and Ensemble Model

In [62]:
print("LightGBM RMSE:", rmse_lgb)
print("XGBoost RMSE:", rmse_xgb)
print("CatBoost RMSE:", rmse_cat)
print("Ensemble RMSE:", rmse_ensemble)

LightGBM RMSE: 381.02332349308665
XGBoost RMSE: 409.05681052950797
CatBoost RMSE: 409.90258177268765
Ensemble RMSE: 382.5243914215874


Comparison of 3 modesl on basis of MSE,MAE,R2_Score

In [69]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

print("=== XGBoost ===")
print("RMSE      :", np.sqrt(mean_squared_error(y_valid, xgb_pred)))
print("MAE       :", mean_absolute_error(y_valid, xgb_pred))
print("R2 Score  :", r2_score(y_valid, xgb_pred))

=== XGBoost ===
RMSE      : 409.05681052950797
MAE       : 123.74895287350516
R2 Score  : 0.9088897761793298


In [70]:
print("=== CatBoost ===")
print("RMSE      :", np.sqrt(mean_squared_error(y_valid, cat_pred)))
print("MAE       :", mean_absolute_error(y_valid, cat_pred))
print("R2 Score  :", r2_score(y_valid, cat_pred))

=== CatBoost ===
RMSE      : 409.90258177268765
MAE       : 145.9822677499419
R2 Score  : 0.9085126252860067


In [71]:
print("=== LightGBM ===")
print("RMSE      :", np.sqrt(mean_squared_error(y_valid, lgb_pred)))
print("MAE       :", mean_absolute_error(y_valid, lgb_pred))
print("R2 Score  :", r2_score(y_valid, lgb_pred))

=== LightGBM ===
RMSE      : 381.02332349308665
MAE       : 134.80283589970645
R2 Score  : 0.9209497987928259


In [72]:


results = {
    "Model"   : ["XGBoost", "CatBoost", "LightGBM"],
    "RMSE"    : [np.sqrt(mean_squared_error(y_valid, xgb_pred)),
                 np.sqrt(mean_squared_error(y_valid, cat_pred)),
                 np.sqrt(mean_squared_error(y_valid, lgb_pred))],
    "MAE"     : [mean_absolute_error(y_valid, xgb_pred),
                 mean_absolute_error(y_valid, cat_pred),
                 mean_absolute_error(y_valid, lgb_pred)],
    "R2 Score": [r2_score(y_valid, xgb_pred),
                 r2_score(y_valid, cat_pred),
                 r2_score(y_valid, lgb_pred)],
}

df_results = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
df_results.index += 1
df_results.index.name = "Rank"
print(df_results)

         Model        RMSE         MAE  R2 Score
Rank                                            
1     LightGBM  381.023323  134.802836  0.920950
2      XGBoost  409.056811  123.748953  0.908890
3     CatBoost  409.902582  145.982268  0.908513


In [75]:
# Step 1: Make sure date features exist in test data
if 'date' in test.columns:
    test['date'] = pd.to_datetime(test['date'])
    
    test['day_of_week'] = test['date'].dt.dayofweek
    test['month'] = test['date'].dt.month
    test['year'] = test['date'].dt.year

# Step 2: Ensure all required features exist
missing_cols = [col for col in features if col not in test.columns]

if len(missing_cols) > 0:
    print("Missing columns:", missing_cols)
else:
    # Step 3: Predictions
    X_test = test[features]

    lgb_pred = lgb_model.predict(X_test)
    xgb_pred = xgb_model.predict(X_test)
    cat_pred = cat_model.predict(X_test)

    # Step 4: Ensemble
    test["predicted_sales"] = (lgb_pred + xgb_pred + cat_pred) / 3

    # Step 5: Show output in notebook (clean table)
    display(test[["id", "predicted_sales"]].head(10))

ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: family: str